# 0.1.Process downloaded metadata (loaddata) to fix index pathing and apply qc
This notebook:
1. Reads the downloaded loaddata CSV files and corrects local image paths.
2. Removes fields missing any required channel TIFF and writes validated `.fixed.parquet` files.
3. Pools whole-image QC across all six plates and gates each channel independently for blur and saturation.
4. Merges QC-passing field identifiers with the validated loaddata, replacing the `.fixed.parquet` files with QC-filtered rows.

In [ ]:
from pathlib import PurePosixPath, Path

import pandas as pd
import pyarrow.dataset as ds    
import numpy as np
from scipy.stats import zscore

from utils.resolve_root import resolve_root

In [3]:
metadata_download_path = resolve_root(globals().get("__file__")) / "metadata"
if not metadata_download_path.exists() or not metadata_download_path.is_dir():
    raise FileNotFoundError(f"Metadata path {metadata_download_path} does not exist or is not a directory")

loaddata_dir = metadata_download_path / "loaddatas"
if not loaddata_dir.exists() or not loaddata_dir.is_dir():
    raise FileNotFoundError(f"Loaddata directory {loaddata_dir} does not exist or is not a directory")
loaddata_files = sorted(
    path for path in loaddata_dir.glob("*.csv") if not path.name.endswith(".fixed.csv")
)
if not loaddata_files:
    raise FileNotFoundError(f"No source loaddata CSV files found in {loaddata_dir}.")

platemap_dir = metadata_download_path / "platemaps"
platemap_files = sorted(
    path for path in platemap_dir.glob("*.csv") if not path.name.endswith(".fixed.csv")
)

barcode_file = metadata_download_path / "Barcode_platemap_pilot_data.csv"
if not barcode_file.exists() or not barcode_file.is_file():
    raise FileNotFoundError(f"Barcode file {barcode_file} does not exist or is not a file")


LOCAL_IMAGE_DIR = Path("~/mnt/bandicoot/PCCMA_data/PedCP90_screen/").expanduser().resolve(strict=True)
SOURCE_PATH_PREFIX = PurePosixPath("/media/18tbdrive/ALSF_pilot_data")
PATH_COL_PREFIX = "PathName_"
FILE_COL_PREFIX = "FileName_"
CHANNELS = ["OrigER", "OrigRNA", "OrigAGP", "OrigMito", "OrigDNA", "OrigBrightfield"]

## Fixing loaddata paths
Assumes local image data exists as the same file structure as that being processed in pediatric cancer atlas profiling repo.  

In [3]:
path_columns = {channel: f"{PATH_COL_PREFIX}{channel}" for channel in CHANNELS}
file_columns = {channel: f"{FILE_COL_PREFIX}{channel}" for channel in CHANNELS}

# Validate every input schema before writing any output, so a malformed plate fails early.
for loaddata_file in loaddata_files:
    csv_columns = set(pd.read_csv(loaddata_file, nrows=0).columns)
    missing_columns_by_channel = {
        channel: [
            column
            for column in (path_columns[channel], file_columns[channel])
            if column not in csv_columns
        ]
        for channel in CHANNELS
    }
    missing_columns_by_channel = {
        channel: columns
        for channel, columns in missing_columns_by_channel.items()
        if columns
    }
    if missing_columns_by_channel:
        missing_details = "; ".join(
            f"{channel}: {', '.join(columns)}"
            for channel, columns in missing_columns_by_channel.items()
        )
        raise KeyError(
            f"Missing anticipated columns in {loaddata_file.name}: {missing_details}"
        )

# Process each loaddata CSV file to validate and fix paths and filenames.
for loaddata_file in loaddata_files:

    loaddata_df = pd.read_csv(loaddata_file)
    print(f"Loaded loaddata file with shape: {loaddata_df.shape}")

    # Collect failed path/file counts
    failure_report = pd.DataFrame(
        0, index=pd.Index(CHANNELS, name="channel"), columns=["path", "file"], dtype=int
    )

    # Remap acquisition-machine directories to the local data root and reject missing paths.
    for channel in CHANNELS:
        path_col = path_columns[channel]
        fixed_paths = []

        for path_cell_content in loaddata_df[path_col]:
            if pd.isna(path_cell_content):
                fixed_paths.append(None)
                failure_report.at[channel, "path"] += 1
                continue

            source_path = PurePosixPath(str(path_cell_content))
            try:
                relative_path = source_path.relative_to(SOURCE_PATH_PREFIX)
            except ValueError:
                fixed_paths.append(None)
                failure_report.at[channel, "path"] += 1
                continue

            fixed_path = LOCAL_IMAGE_DIR.joinpath(*relative_path.parts)
            fixed_paths.append(str(fixed_path))

        loaddata_df[path_col] = fixed_paths

    # Remove invalid directories first; only surviving rows can form meaningful TIFF paths.
    rows_before_path_drop = len(loaddata_df)
    loaddata_df = loaddata_df.dropna(subset=list(path_columns.values())).copy()
    print(f"Dropped {rows_before_path_drop - len(loaddata_df)} rows with missing paths")

    # Pair each channel's validated directory and filename to detect missing TIFF transfers.
    all_channel_files_exist = pd.Series(True, index=loaddata_df.index)
    for channel in CHANNELS:
        path_col = path_columns[channel]
        file_col = file_columns[channel]
        channel_files_exist = pd.Series(False, index=loaddata_df.index)

        for row_index, (path_cell_content, file_cell_content) in loaddata_df[
            [path_col, file_col]
        ].iterrows():
            if pd.isna(file_cell_content):
                continue

            fixed_file = Path(path_cell_content) / str(file_cell_content)
            channel_files_exist.at[row_index] = fixed_file.is_file()

        failure_report.at[channel, "file"] = int((~channel_files_exist).sum())
        all_channel_files_exist &= channel_files_exist

    # Keep only rows that are complete across every anticipated imaging channel.
    rows_before_file_drop = len(loaddata_df)
    loaddata_df = loaddata_df.loc[all_channel_files_exist].copy()
    print(f"Dropped {rows_before_file_drop - len(loaddata_df)} rows with missing TIFF files")
    print("\nFailed validation counts by channel:")
    print(failure_report.to_string())

    output_file = loaddata_file.with_name(f"{loaddata_file.stem}.fixed.parquet")
    loaddata_df.to_parquet(output_file, index=False)
    print(f"Saved {len(loaddata_df)} fully validated rows to {output_file}\n")

Loaded loaddata file with shape: (2160, 25)
Dropped 0 rows with missing paths
Dropped 0 rows with missing TIFF files

Failed validation counts by channel:
                 path  file
channel                    
OrigER              0     0
OrigRNA             0     0
OrigAGP             0     0
OrigMito            0     0
OrigDNA             0     0
OrigBrightfield     0     0
Saved 2160 fully validated rows to /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143976_concatenated.fixed.parquet

Loaded loaddata file with shape: (1259, 25)
Dropped 0 rows with missing paths
Dropped 0 rows with missing TIFF files

Failed validation counts by channel:
                 path  file
channel                    
OrigER              0     0
OrigRNA             0     0
OrigAGP             0     0
OrigMito            0     0
OrigDNA             0     0
OrigBrightfield     0     0
Saved 1259 fully validated rows to /home/weishanli/Waylab/virtual_staining_metric_analysis/me

## Gate whole-image QC before merging with validated loaddata
Pool the six QC CSVs before calculating Z-scores **separately for each of the six channels** (population standard deviation, `ddof=0`). Reject an entire field if any channel has `abs(blur_z) > 2.5` or `abs(saturation_z) > 2`. Both gates are calculated on the original pooled QC table, not sequentially on filtered subsets.

`Metadata_Reimaged` is used only to select example outliers for inspection; it does not exempt any field from filtering. Constant metric columns have no statistical outliers. Missing/nonfinite metrics or ambiguous field keys stop processing rather than silently deciding QC status.

The following cells can also be run directly on existing validated Parquet files after running the import and pathing cells. Re-running the path-validation cell first rebuilds the unfiltered validated inputs from the source CSVs.

In [4]:
qc_dir = metadata_download_path / "whole_img_qc_output"
qc_files = [qc_dir / f"BR{number:08d}.csv" for number in range(143976, 143982)]
missing_qc_files = [str(path) for path in qc_files if not path.is_file()]
if missing_qc_files:
    raise FileNotFoundError(f"Missing QC inputs: {missing_qc_files}")

field_keys = ["Metadata_Plate", "Metadata_Well", "Metadata_Site"]
qc_channels = ["OrigER", "OrigAGP", "OrigDNA", "OrigRNA", "OrigMito", "OrigBrightfield"]
blur_threshold_z = 2.5
saturation_threshold_z = 2.0
qc_df = pd.concat([pd.read_csv(path) for path in qc_files], ignore_index=True)
required_qc_columns = field_keys + ["Metadata_Reimaged"] + [
    f"{metric}_{channel}"
    for metric in ("ImageQuality_PowerLogLogSlope", "ImageQuality_PercentMaximal")
    for channel in qc_channels
]
missing_columns = sorted(set(required_qc_columns) - set(qc_df.columns))
if missing_columns:
    raise KeyError(f"Missing QC columns: {missing_columns}")
if qc_df[field_keys].isna().any().any() or qc_df.duplicated(field_keys).any():
    raise ValueError("QC must have one non-null, unique plate-well-site key per field.")

blur_failures = pd.DataFrame(False, index=qc_df.index, columns=qc_channels)
saturation_failures = blur_failures.copy()
outlier_tables = {"blur": [], "saturation": []}
for gate, metric, threshold, failures in (
    ("blur", "ImageQuality_PowerLogLogSlope", blur_threshold_z, blur_failures),
    ("saturation", "ImageQuality_PercentMaximal", saturation_threshold_z, saturation_failures),
):
    for channel in qc_channels:
        column = f"{metric}_{channel}"
        values = pd.to_numeric(qc_df[column], errors="raise")
        if not np.isfinite(values.to_numpy()).all():
            raise ValueError(f"Nonfinite QC values in {column}; QC status is undefined.")
        scores = np.zeros(len(values)) if values.nunique() == 1 else zscore(values, ddof=0)
        failures[channel] = np.abs(scores) > threshold
        outliers = qc_df.loc[failures[channel], field_keys + ["Metadata_Reimaged"]].copy()
        outliers[metric] = values.loc[failures[channel]]
        outliers["Channel"] = channel
        outliers["Z_score"] = scores[failures[channel].to_numpy()]
        outlier_tables[gate].append(outliers)

blur_outliers = pd.concat(outlier_tables["blur"], ignore_index=True)
saturation_outliers = pd.concat(outlier_tables["saturation"], ignore_index=True)
blur_failed_fields = blur_failures.any(axis=1)
saturation_failed_fields = saturation_failures.any(axis=1)
qc_failed_fields = blur_failed_fields | saturation_failed_fields
qc_passing_fields = qc_df.loc[~qc_failed_fields, field_keys].copy()
qc_failure_counts = pd.DataFrame({
    "blur": blur_failures.sum(),
    "saturation": saturation_failures.sum(),
    "either": (blur_failures | saturation_failures).sum(),
})
qc_failure_counts.index.name = "Channel"
display(qc_failure_counts)
for label, failed in (
    ("Blur", blur_failed_fields),
    ("Saturation", saturation_failed_fields),
    ("Either gate", qc_failed_fields),
):
    print(f"{label}: {failed.sum():,}/{len(qc_df):,} unique fields removed "
          f"({100 * failed.mean():.2f}%)")

blur_outliers_filtered = blur_outliers.loc[~blur_outliers["Metadata_Reimaged"]]
saturation_outliers_filtered = saturation_outliers.loc[~saturation_outliers["Metadata_Reimaged"]]

,blur,saturation,either
Channel,,,
OrigER,140,32,171
OrigAGP,217,26,243
OrigDNA,167,49,213
OrigRNA,133,136,267
OrigMito,16,172,188
OrigBrightfield,84,1,85


Blur: 409/10,248 unique fields removed (3.99%)
Saturation: 354/10,248 unique fields removed (3.45%)
Either gate: 752/10,248 unique fields removed (7.34%)


## Merge passing fields into validated Parquet files
Inner-join the post-threshold QC field keys to each validated loaddata file without adding QC columns or duplicating rows. Validate all input keys and QC coverage before replacing any output. Fields lacking QC records stop processing rather than being silently discarded. The fixed Parquet filenames stay unchanged for downstream notebooks.

In [5]:
fixed_loaddata_files = sorted(loaddata_dir.glob("*.fixed.parquet"))
if not fixed_loaddata_files:
    raise FileNotFoundError(f"No validated loaddata Parquet files found in {loaddata_dir}.")

# Preflight every merge before overwriting any validated output.
qc_filtered_outputs = {}
qc_merge_report = []
for fixed_file in fixed_loaddata_files:
    validated_df = pd.read_parquet(fixed_file)
    missing_keys = sorted(set(field_keys) - set(validated_df.columns))
    if missing_keys:
        raise KeyError(f"Missing field keys in {fixed_file.name}: {missing_keys}")
    if validated_df[field_keys].isna().any().any() or validated_df.duplicated(field_keys).any():
        raise ValueError(f"Non-null unique field keys required in {fixed_file.name}.")
    coverage = validated_df[field_keys].merge(
        qc_df[field_keys], on=field_keys, how="left", validate="one_to_one", indicator=True
    )
    unmatched = coverage.loc[coverage["_merge"] == "left_only", field_keys]
    if not unmatched.empty:
        raise ValueError(
            f"{fixed_file.name}: {len(unmatched)} fields have no QC record. "
            f"Examples: {unmatched.head().to_dict('records')}"
        )
    filtered_df = validated_df.merge(
        qc_passing_fields, on=field_keys, how="inner", validate="one_to_one", sort=False
    ).loc[:, validated_df.columns]
    qc_filtered_outputs[fixed_file] = filtered_df
    removed_count = len(validated_df) - len(filtered_df)
    qc_merge_report.append({
        "file": fixed_file.name,
        "validated_fields": len(validated_df),
        "qc_removed_fields": removed_count,
        "retained_fields": len(filtered_df),
        "percentage_removed": 100 * removed_count / len(validated_df) if len(validated_df) else 0,
    })

for fixed_file, filtered_df in qc_filtered_outputs.items():
    # Stage each write so a failed Parquet serialization leaves the original intact.
    temporary_file = fixed_file.with_suffix(".parquet.tmp")
    try:
        filtered_df.to_parquet(temporary_file, index=False)
        temporary_file.replace(fixed_file)
    finally:
        temporary_file.unlink(missing_ok=True)
    print(f"Saved {len(filtered_df):,} QC-passing fields to {fixed_file}")

qc_merge_report = pd.DataFrame(qc_merge_report)
display(qc_merge_report)

Saved 2,038 QC-passing fields to /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143976_concatenated.fixed.parquet
Saved 1,196 QC-passing fields to /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143977_concatenated.fixed.parquet
Saved 2,033 QC-passing fields to /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143978_concatenated.fixed.parquet
Saved 1,166 QC-passing fields to /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143979_concatenated.fixed.parquet
Saved 1,888 QC-passing fields to /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143980_concatenated.fixed.parquet
Saved 1,175 QC-passing fields to /home/weishanli/Waylab/virtual_staining_metric_analysis/metadata/loaddatas/BR00143981_concatenated.fixed.parquet


,file,validated_fields,qc_removed_fields,retained_fields,percentage_removed
0,BR00143976_concatenated.fixed.parquet,2160,122,2038,5.648148
1,BR00143977_concatenated.fixed.parquet,1259,63,1196,5.003971
2,BR00143978_concatenated.fixed.parquet,2160,127,2033,5.879630
3,BR00143979_concatenated.fixed.parquet,1259,93,1166,7.386815
4,BR00143980_concatenated.fixed.parquet,2160,272,1888,12.592593
5,BR00143981_concatenated.fixed.parquet,1250,75,1175,6.000000


In [6]:
# Read back outputs and independently confirm that no failed field survived.
failed_field_index = pd.MultiIndex.from_frame(qc_df.loc[qc_failed_fields, field_keys])
for fixed_file, expected_df in qc_filtered_outputs.items():
    saved_df = pd.read_parquet(fixed_file)
    pd.testing.assert_frame_equal(saved_df, expected_df)
    saved_field_index = pd.MultiIndex.from_frame(saved_df[field_keys])
    assert not saved_field_index.isin(failed_field_index).any(), fixed_file.name

print(f"Verified {len(qc_filtered_outputs)} QC-filtered Parquet files; "
      f"retained {qc_merge_report['retained_fields'].sum():,} fields and "
      f"removed {qc_merge_report['qc_removed_fields'].sum():,}.")

Verified 6 QC-filtered Parquet files; retained 9,496 fields and removed 752.


## Curate metadata to map well to cell line and plate identity

In [4]:
barcode = pd.read_csv(barcode_file)
metadata_collect = []
for index, row in barcode.iterrows():
    (barcode, time_point, platemap) = (row["barcode"], row["time_point"], row["platemap_file"])
    platemap_file = platemap_dir / f"{platemap}.csv"
    if not platemap_file.exists() or not platemap_file.is_file():
        raise FileNotFoundError(f"Platemap file {platemap_file} does not exist or is not a file")

    metadata_collect.append(pd.read_csv(platemap_file).assign(Metadata_Plate=barcode, time_point=time_point))

metadata = pd.concat(metadata_collect, ignore_index=True).rename(columns={"well": "Metadata_Well"})
metadata

,cell_line,row,column,Metadata_Well,seeding_density,condition,Metadata_Plate,time_point
0,DAOY,C,3,C03,1000,standard,BR00143976,24
1,DAOY,D,3,D03,1000,standard,BR00143976,24
2,IMR32,E,3,E03,1000,standard,BR00143976,24
3,IMR32,F,3,F03,1000,standard,BR00143976,24
4,PA1,G,3,G03,1000,standard,BR00143976,24
...,...,...,...,...,...,...,...,...
1135,U2-OS,N,20,N20,8000,standard,BR00143981,72
1136,U2-OS,M,21,M21,12000,standard,BR00143981,72
1137,U2-OS,N,21,N21,12000,standard,BR00143981,72
1138,U2-OS,M,22,M22,12000,standard,BR00143981,72


## Separate loaddata CSV for degradation and model training

The degradation export uses the curated `metadata` from cell 12, restricted to `time_point == 24` hours; plate IDs are derived from that table rather than hard-coded.

All QC-filtered `*.fixed.parquet` files are opened as one logical, concatenated PyArrow dataset. The plate selection is pushed into the Arrow scan before conversion to pandas, allowing Parquet statistics to skip irrelevant row groups where possible and avoiding pandas materialization of other time points. Original loaddata columns are retained.

An inner join on **both plate and well** then retains only annotated 24-hour wells and adds all curated annotations, including `cell_line`, `seeding_density`, `condition`, and `time_point`. A many-to-one validation prevents duplicate metadata keys from multiplying imaging-site rows; sites already removed by path validation or QC cannot be restored by this join.

Running the next cell **overwrites** `metadata/loaddata_for_degradation.csv` without a CSV index and reports exported fields, plate count, and fields excluded for unmatched wells. The fixed Parquet inputs and the full `metadata` table remain unchanged.

In [ ]:
# Use the curated plate/well metadata, not hard-coded plate IDs or filenames.
metadata_24h = metadata.loc[metadata["time_point"].eq(24)].copy()
metadata_keys = ["Metadata_Plate", "Metadata_Well"]
if metadata_24h.empty:
    raise ValueError("No 24-hour rows found in the curated metadata.")
if metadata_24h[metadata_keys].isna().any().any() or metadata_24h.duplicated(metadata_keys).any():
    raise ValueError("24-hour metadata must have non-null, unique plate/well keys.")

# Treat all fixed Parquets as one logical concatenated dataset. Push the plate
 # selections into the scan so irrelevant rows never become pandas objects.
fixed_loaddata_files = sorted(loaddata_dir.glob("*.fixed.parquet"))
if not fixed_loaddata_files:
    raise FileNotFoundError(f"No fixed loaddata Parquet files found in {loaddata_dir}.")
loaddata_dataset = ds.dataset(fixed_loaddata_files, format="parquet")
loaddata_24h = loaddata_dataset.to_table(
    filter=ds.field("Metadata_Plate").isin(metadata_24h["Metadata_Plate"].unique())
).to_pandas()

# Match exact plate/well pairs and attach every curated annotation to each site.
degradation_df = loaddata_24h.merge(
    metadata_24h, on=metadata_keys, how="inner", validate="many_to_one", sort=False
)
if degradation_df.empty:
    raise ValueError("No QC-passing loaddata fields match the 24-hour metadata.")

degradation_csv = metadata_download_path / "loaddata_for_degradation.csv"
degradation_df.to_csv(degradation_csv, index=False)

In [ ]:
# model training split TODO